In [1]:
import os
import random
import numpy as np
import copy
from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageFilter

# ================= CONFIG =================
IMG_SIZE = 32
ALPHABET = "йклмнопрсту"
NUM_SAMPLES_PER_LETTER = 1000
DATASET_DIR = "dataset_images"

# ================= ВАРИАНТ: tanh, alpha=3, T0=0 =================
ALPHA = 3
T0 = 0


In [2]:
# ================= ГЕНЕРАЦИЯ ДАННЫХ =================
def get_local_fonts():
    font_paths = [f for f in os.listdir('.') if f.endswith('.ttf')]
    if not font_paths:
        sys_path = "C:\\Windows\\Fonts\\"
        common_fonts = ['arial.ttf', 'times.ttf', 'verdana.ttf']
        font_paths = [os.path.join(sys_path, f) for f in common_fonts if os.path.exists(os.path.join(sys_path, f))]
    if not font_paths:
        raise RuntimeError("Файлы .ttf не найдены в папке проекта!")
    return font_paths

def generate_char_image(char, font_path, save_path=None):
    canvas_size = IMG_SIZE * 3
    bg_color = random.randint(235, 255)
    img = Image.new('L', (canvas_size, canvas_size), bg_color)
    draw = ImageDraw.Draw(img)

    font_size = random.randint(int(IMG_SIZE * 0.9), int(IMG_SIZE * 1.3))
    font = ImageFont.truetype(font_path, font_size)

    bbox = draw.textbbox((0, 0), char, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (canvas_size - w) / 2 - bbox[0]
    y = (canvas_size - h) / 2 - bbox[1]
    draw.text((x, y), char, font=font, fill=0)

    img = img.rotate(random.uniform(-15, 15), resample=Image.BICUBIC, fillcolor=bg_color)
    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 0.8)))

    center = canvas_size // 2
    s = IMG_SIZE // 2
    jx, jy = random.randint(-2, 2), random.randint(-2, 2)
    img = img.crop((center - s + jx, center - s + jy, center + s + jx, center + s + jy))

    if save_path:
        img.save(save_path)

    img = ImageOps.invert(img)
    arr = np.array(img).astype(np.float32) / 255.0
    if random.random() < 0.4:
        arr += np.random.normal(0, 0.01, arr.shape)

    arr = (np.clip(arr, 0, 1) > 0.4).astype(np.float32)
    return arr.flatten()

def generate_dataset():
    fonts = get_local_fonts()
    all_images = []
    all_labels = []

    if not os.path.exists(DATASET_DIR):
        os.makedirs(DATASET_DIR)
        print(f"Создана директория: {DATASET_DIR}")

    print("Генерация данных и сохранение изображений...")
    for label, char in enumerate(ALPHABET):
        char_dir = os.path.join(DATASET_DIR, char)
        if not os.path.exists(char_dir):
            os.makedirs(char_dir)

        for i in range(NUM_SAMPLES_PER_LETTER):
            try:
                f = random.choice(fonts)
                c = char.upper() if random.random() < 0.3 else char
                img_name = f"img_{i}.png"
                save_path = os.path.join(char_dir, img_name)
                img_vector = generate_char_image(c, f, save_path=save_path)
                all_images.append(img_vector)
                all_labels.append(label)
            except Exception as e:
                continue

    print(f"Генерация завершена. Всего сохранено {len(all_images)} изображений.")
    return np.array(all_images), np.array(all_labels)

# Запуск генерации
X, y = generate_dataset()


Создана директория: dataset_images
Генерация данных и сохранение изображений...
Генерация завершена. Всего сохранено 11000 изображений.


In [3]:
# ================= МОЯ РЕАЛИЗАЦИЯ (tanh, alpha=3, T0=0) =================
class Network:
    def __init__(self, sizes):
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x) * np.sqrt(2 / x) for x, y in zip(sizes[:-1], sizes[1:])]
        self.history = []

    def tanh(self, z):
        alpha = 3
        T0 = 0
        return np.tanh(alpha * (z - T0))

    def tanh_prime(self, z):
        alpha = 3
        tanh_val = self.tanh(z)
        return alpha * (1 - tanh_val ** 2)

    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
        return exp_z / np.sum(exp_z, axis=0, keepdims=True)

    def feedforward(self, a):
        for i, (b, w) in enumerate(zip(self.biases, self.weights)):
            z = np.dot(w, a) + b
            a = self.softmax(z) if i == self.num_layers - 2 else self.tanh(z)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta,
            test_data=None, early_stopping=True, patience=5, monitor='val_loss'):
        n = len(training_data)
        if test_data:
            n_test = len(test_data)

        best_weights = copy.deepcopy(self.weights)
        best_biases = copy.deepcopy(self.biases)
        best_metric = float('inf') if monitor == 'val_loss' else 0
        wait = 0

        for epoch in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k + mini_batch_size] for k in range(0, n, mini_batch_size)]

            for mini_batch in mini_batches:
                x_batch = np.hstack([x for x, y in mini_batch])
                y_batch = np.hstack([y for x, y in mini_batch])
                self.update_mini_batch(x_batch, y_batch, eta)

            train_acc = self.evaluate([(x, np.argmax(y)) for x, y in training_data]) / n
            train_loss = self.compute_loss(training_data)

            val_acc = val_loss = None
            if test_data:
                val_acc = self.evaluate(test_data) / n_test
                val_loss = self.compute_loss(test_data, is_test=True)

            log = {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc
            }
            self.history.append(log)

            print_str = f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f}"
            if test_data:
                print_str += f" - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
            print(print_str)

            if early_stopping and test_data:
                current_metric = val_loss if monitor == 'val_loss' else val_acc
                is_improved = (current_metric < best_metric) if monitor == 'val_loss' else (current_metric > best_metric)

                if is_improved:
                    best_metric = current_metric
                    best_weights = copy.deepcopy(self.weights)
                    best_biases = copy.deepcopy(self.biases)
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f"Early stopping triggered at epoch {epoch+1}. Best {monitor}: {best_metric:.4f}")
                        break

        self.weights = best_weights
        self.biases = best_biases

    def update_mini_batch(self, x_batch, y_batch, eta):
        nabla_b, nabla_w = self.backprop(x_batch, y_batch)
        batch_size = x_batch.shape[1]
        self.weights = [w - (eta / batch_size) * nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta / batch_size) * nb for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x_batch, y_batch):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        activation = x_batch
        activations = [x_batch]
        zs = []

        for i, (b, w) in enumerate(zip(self.biases, self.weights)):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = self.softmax(z) if i == self.num_layers - 2 else self.tanh(z)
            activations.append(activation)

        delta = activations[-1] - y_batch
        nabla_b[-1] = np.sum(delta, axis=1, keepdims=True)
        nabla_w[-1] = np.dot(delta, activations[-2].T)

        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = self.tanh_prime(z)
            delta = np.dot(self.weights[-l + 1].T, delta) * sp
            nabla_b[-l] = np.sum(delta, axis=1, keepdims=True)
            nabla_w[-l] = np.dot(delta, activations[-l - 1].T)

        return nabla_b, nabla_w

    def evaluate(self, test_data):
        test_results = [(np.argmax(self.feedforward(x)), y) for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)

    def compute_loss(self, data, is_test=False):
        total_loss = 0.0
        for x, y in data:
            if is_test:
                y = self.vectorize_label(y)
            output = self.feedforward(x)
            loss = -np.sum(y * np.log(output + 1e-8))
            total_loss += loss
        return total_loss / len(data)

    def vectorize_label(self, label):
        e = np.zeros((self.sizes[-1], 1))
        e[label] = 1.0
        return e


In [7]:
# ================= ОБУЧЕНИЕ МОЕЙ СЕТИ =================
num_classes = len(ALPHABET)
test_size = 0.2
n_samples = len(X)
indices = np.random.permutation(n_samples)
test_count = int(test_size * n_samples)

test_indices = indices[:test_count]
train_indices = indices[test_count:]

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

def to_categorical(labels, num_classes):
    result = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        result[i, label] = 1.0
    return result

y_train_cat = to_categorical(y_train, num_classes)

X_train = [x.reshape(-1, 1) for x in X_train]
X_test = [x.reshape(-1, 1) for x in X_test]
y_train = [y.reshape(-1, 1) for y in y_train_cat]
training_data = list(zip(X_train, y_train))
test_data = list(zip(X_test, y_test))

input_size = IMG_SIZE * IMG_SIZE
# net = Network([input_size, 128, 64, num_classes])
# print("\n=== ОБУЧЕНИЕ МОЕЙ СЕТИ (tanh, alpha=3, T0=0) ===")
# net.SGD(training_data, epochs=100, mini_batch_size=128, eta=0.1, 
#         test_data=test_data, early_stopping=True, patience=10, monitor='val_loss')

# print(f"\nОбучение завершено. Лучший val_loss: {min([h['val_loss'] for h in net.history if h['val_loss'] is not None]):.4f}")
# print(f"Лучшая val_acc: {max([h['val_acc'] for h in net.history if h['val_acc'] is not None]):.4f}")


In [5]:
# # # ================= РЕАЛИЗАЦИЯ НА Keras =================
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers
# import numpy as np

# # 0. Подготовка данных (разбиение на train/test)
# num_classes = len(ALPHABET)
# test_size = 0.2
# n_samples = len(X)
# indices = np.random.permutation(n_samples)
# test_count = int(test_size * n_samples)

# test_indices = indices[:test_count]
# train_indices = indices[test_count:]

# X_train, X_test = X[train_indices], X[test_indices]
# y_train, y_test = y[train_indices], y[test_indices]

# # 1. Подготовка данных для Keras
# # Используем индексы (0, 1, 2, ...), а не One-Hot!
# y_train_idx = np.array([np.argmax(y) for y in y_train], dtype=np.int64)
# y_test_idx = np.array([np.argmax(y) for y in y_test], dtype=np.int64)

# # Данные оставляем 0/1, формируем правильную форму для Keras (N, IMG_SIZE, IMG_SIZE, 1)
# X_train_k = np.array([x.reshape(IMG_SIZE, IMG_SIZE, 1) for x in X_train]).astype('float32')
# X_test_k = np.array([x.reshape(IMG_SIZE, IMG_SIZE, 1) for x in X_test]).astype('float32')

# # 2. Архитектура модели (аналогично моей сети: input -> 128 -> 64 -> num_classes)
# model = keras.Sequential([
#     layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
#     layers.Flatten(),  # Выпрямляем изображение в вектор
    
#     # Слой 1: Xavier Uniform + Tanh
#     layers.Dense(128, kernel_initializer='glorot_uniform'),
#     layers.Activation('tanh'),
    
#     # Слой 2: Xavier Uniform + Tanh
#     layers.Dense(64, kernel_initializer='glorot_uniform'),
#     layers.Activation('tanh'),
    
#     # Выходной слой: Softmax для многоклассовой классификации
#     layers.Dense(num_classes, activation='softmax')
# ])

# # 3. Компиляция
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=0.001),
#     loss='sparse_categorical_crossentropy', 
#     metrics=['accuracy']
# )

# # 4. Коллбэки (Scheduler + EarlyStopping)
# callbacks = [
#     keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss', factor=0.5, patience=7, min_lr=0.00001
#     ),
#     keras.callbacks.EarlyStopping(
#         monitor='val_accuracy', patience=20, restore_best_weights=True, mode='max'
#     )
# ]

# # 5. Обучение
# print("\n=== ОБУЧЕНИЕ KERAS МОДЕЛИ (tanh) ===")
# history = model.fit(
#     X_train_k, y_train_idx,
#     epochs=100,
#     batch_size=32,
#     validation_data=(X_test_k, y_test_idx),
#     shuffle=True,
#     callbacks=callbacks,
#     verbose=1
# )

# # 6. Проверка на тестовой выборке
# test_loss, test_acc = model.evaluate(X_test_k, y_test_idx, verbose=0)
# print(f"\nKeras модель: test_loss = {test_loss:.4f}, test_acc = {test_acc:.4f}")

# # 7. Функция для предсказания на новых изображениях
# def predict_image(model, img_path, alphabet=ALPHABET):
#     """
#     Предсказывает букву на изображении.
    
#     Параметры:
#         model: обученная Keras модель
#         img_path: путь к изображению или PIL.Image объект
#         alphabet: алфавит классов
    
#     Возвращает:
#         sorted_results: список кортежей (буква, вероятность) отсортированный по убыванию
#     """
#     from PIL import Image, ImageOps
    
#     # Загрузка изображения
#     if isinstance(img_path, str):
#         img = Image.open(img_path).convert('L')
#     else:
#         img = img_path.convert('L')
    
#     # Инвертирование и поиск границ
#     inv_img = ImageOps.invert(img)
#     bbox = inv_img.getbbox()
#     if not bbox:
#         return [(c, 0.0) for c in alphabet]
    
#     # Кроп и добавление отступов
#     letter = inv_img.crop(bbox)
#     w, h = letter.size
#     max_dim = max(w, h)
#     pad = int(max_dim * 0.3)
#     container = Image.new('L', (max_dim + pad, max_dim + pad), 0)
#     container.paste(letter, ((max_dim + pad - w)//2, (max_dim + pad - h)//2))
    
#     # Ресайз до IMG_SIZE x IMG_SIZE
#     letter_resized = container.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS)
    
#     # Бинаризация как при генерации данных
#     img_array = np.array(letter_resized).astype(np.float32) / 255.0
#     img_array = (img_array > 0.35).astype(np.float32)
    
#     # Предсказание
#     img_input = img_array.reshape(1, IMG_SIZE, IMG_SIZE, 1)
#     output = model.predict(img_input, verbose=0)[0]
    
#     results = [(alphabet[i], float(prob)) for i, prob in enumerate(output)]
#     return sorted(results, key=lambda x: x[1], reverse=True)

# # Пример использования:
# # result = predict_image(model, 'dataset_images/а/img_0.png')
# # print(f"Предсказание: {result[0][0]} с вероятностью {result[0][1]*100:.2f}%")


In [8]:
# ================= РЕАЛИЗАЦИЯ НА PYTORCH =================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Проверка доступности CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

# PyTorch модель с tanh активацией
class PyTorchNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(PyTorchNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.tanh = nn.Tanh()
        # Инициализация весов
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        x = self.tanh(self.fc1(x))
        x = self.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

# Подготовка данных для PyTorch - данные уже бинарные 0/1
X_train_arr = np.array([x.flatten() for x in X_train])
X_test_arr = np.array([x.flatten() for x in X_test])
y_train_indices = np.array([np.argmax(y) for y in y_train], dtype=np.int64)
y_test_indices = np.array([np.argmax(y) for y in y_test], dtype=np.int64)

X_train_pt = torch.FloatTensor(X_train_arr).to(device)
y_train_pt = torch.LongTensor(y_train_indices).to(device)
X_test_pt = torch.FloatTensor(X_test_arr).to(device)
y_test_pt = torch.LongTensor(y_test_indices).to(device)

train_dataset = TensorDataset(X_train_pt, y_train_pt)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Создание модели
pt_model = PyTorchNet(input_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, min_lr=0.00001)

# Обучение
print("\n=== ОБУЧЕНИЕ PYTORCH МОДЕЕЛИ (tanh) ===")
best_val_loss = float('inf')
patience = 20
wait = 0

for epoch in range(100):
    pt_model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = pt_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    # Валидация
    pt_model.eval()
    with torch.no_grad():
        val_outputs = pt_model(X_test_pt)
        val_loss = criterion(val_outputs, y_test_pt).item()
        _, val_predicted = torch.max(val_outputs.data, 1)
        val_acc = (val_predicted == y_test_pt).sum().item() / len(y_test_pt)
    
    # Обновление learning rate
    scheduler.step(val_loss)
    
    train_acc = correct / total
    print(f"Epoch {epoch+1}/100 - loss: {total_loss/len(train_loader):.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")
    
    # Early stopping по accuracy
    if val_acc > 0.95:  # Если достигли хорошей точности
        print(f"Достигнута хорошая точность: {val_acc:.4f}")
        best_val_loss = val_loss
        break
    
    if val_loss < best_val_loss - 0.001:
        best_val_loss = val_loss
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}. Best val_loss: {best_val_loss:.4f}")
            break

print(f"\nPyTorch модель: лучший val_loss = {best_val_loss:.4f}")


Используемое устройство: cpu

=== ОБУЧЕНИЕ PYTORCH МОДЕЕЛИ (tanh) ===
Epoch 1/100 - loss: 0.5262 - acc: 0.8570 - val_loss: 5.6597 - val_acc: 0.0845
Epoch 2/100 - loss: 0.1152 - acc: 0.9743 - val_loss: 6.7027 - val_acc: 0.0886
Epoch 3/100 - loss: 0.0529 - acc: 0.9908 - val_loss: 7.3999 - val_acc: 0.0895
Epoch 4/100 - loss: 0.0248 - acc: 0.9975 - val_loss: 8.2663 - val_acc: 0.0895
Epoch 5/100 - loss: 0.0152 - acc: 0.9984 - val_loss: 8.9346 - val_acc: 0.0882
Epoch 6/100 - loss: 0.0171 - acc: 0.9962 - val_loss: 8.9932 - val_acc: 0.0886
Epoch 7/100 - loss: 0.0085 - acc: 0.9990 - val_loss: 9.8971 - val_acc: 0.0845
Epoch 8/100 - loss: 0.0104 - acc: 0.9977 - val_loss: 9.8139 - val_acc: 0.0891
Epoch 9/100 - loss: 0.0209 - acc: 0.9938 - val_loss: 9.9551 - val_acc: 0.0859
Epoch 10/100 - loss: 0.0052 - acc: 0.9997 - val_loss: 10.2310 - val_acc: 0.0877
Epoch 11/100 - loss: 0.0020 - acc: 1.0000 - val_loss: 10.3984 - val_acc: 0.0886
Epoch 12/100 - loss: 0.0015 - acc: 1.0000 - val_loss: 10.5004 - val_

In [11]:
import pickle

# ================= СОХРАНЕНИЕ МОДЕЛЕЙ =================
def save_model(net, path='model.pkl'):
    with open(path, 'wb') as f:
        pickle.dump({'weights': net.weights, 'biases': net.biases, 'sizes': net.sizes}, f)
    print(f"Модель сохранена в {path}")

# # Сохраняем мою модель (если обучали)
# save_model(net, 'model_my.pkl')

# Сохраняем Keras модель
# model.save('model_keras.h5')
# print("Keras модель сохранена в model_keras.h5")

# # Сохраняем PyTorch модель (если обучали)
torch.save(pt_model.state_dict(), 'model_pytorch.pth')
print("PyTorch модель сохранена в model_pytorch.pth")

print("\n=== СРАВНЕНИЕ РЕАЛИЗАЦИЙ ===")
# # Моя сеть (если обучали)
# print(f"Моя сеть (tanh, α=3, T0=0): val_acc = {max([h['val_acc'] for h in net.history if h['val_acc'] is not None]):.4f}")
# Keras модель
# print(f"Keras (tanh): test_acc = {test_acc:.4f}")
# # PyTorch (если обучали)
# print(f"PyTorch (tanh): лучший val_loss = {best_val_loss:.4f}")


PyTorch модель сохранена в model_pytorch.pth

=== СРАВНЕНИЕ РЕАЛИЗАЦИЙ ===


In [12]:
# ================= УНИВЕРСАЛЬНАЯ РИСОВАЛКА ДЛЯ ВСЕХ МОДЕЛЕЙ =================
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageDraw, ImageTk, ImageOps
from tensorflow import keras
import pickle
import numpy as np

# --- Предсказатель для моей модели ---
class MyNetworkPredictor:
    def __init__(self, weights, biases, sizes, alphabet):
        self.weights = weights
        self.biases = biases
        self.sizes = sizes
        self.alphabet = alphabet
        self.num_layers = len(sizes)
        self.alpha = 3
        self.T0 = 0
    
    def tanh(self, z):
        return np.tanh(self.alpha * (z - self.T0))
    
    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
        return exp_z / np.sum(exp_z, axis=0, keepdims=True)
    
    def feedforward(self, a):
        for i, (b, w) in enumerate(zip(self.biases, self.weights)):
            z = np.dot(w, a) + b
            a = self.softmax(z) if i == self.num_layers - 2 else self.tanh(z)
        return a
    
    def predict_all(self, processed_img):
        img_vector = processed_img.reshape(-1, 1)
        output = self.feedforward(img_vector).flatten()
        results = [(self.alphabet[i], prob) for i, prob in enumerate(output)]
        return sorted(results, key=lambda x: x[1], reverse=True)

# --- Предсказатель для Keras ---
class KerasPredictor:
    def __init__(self, model, alphabet):
        self.model = model
        self.alphabet = alphabet
    
    def predict_all(self, processed_img):
        # processed_img уже имеет форму (IMG_SIZE, IMG_SIZE)
        img = processed_img.reshape(1, IMG_SIZE, IMG_SIZE, 1)
        output = self.model.predict(img, verbose=0)[0]
        results = [(self.alphabet[i], float(prob)) for i, prob in enumerate(output)]
        return sorted(results, key=lambda x: x[1], reverse=True)

# --- Предсказатель для PyTorch (если используется) ---
try:
    import torch
    import torch.nn as nn
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_size = IMG_SIZE * IMG_SIZE
    
    class PyTorchNet(nn.Module):
        def __init__(self, input_size, num_classes):
            super(PyTorchNet, self).__init__()
            self.fc1 = nn.Linear(input_size, 128)
            self.fc2 = nn.Linear(128, 64)
            self.fc3 = nn.Linear(64, num_classes)
            self.tanh = nn.Tanh()

        def forward(self, x):
            x = self.tanh(self.fc1(x))
            x = self.tanh(self.fc2(x))
            x = self.fc3(x)
            return x
    
    class PyTorchPredictor:
        def __init__(self, model, alphabet, device):
            self.model = model
            self.alphabet = alphabet
            self.device = device
            self.model.eval()
        
        def predict_all(self, processed_img):
            img = torch.FloatTensor(processed_img.flatten()).unsqueeze(0).to(self.device)
            with torch.no_grad():
                output = torch.softmax(self.model(img), dim=1)[0].cpu().numpy()
            results = [(self.alphabet[i], float(prob)) for i, prob in enumerate(output)]
            return sorted(results, key=lambda x: x[1], reverse=True)
    
    PYTORCH_AVAILABLE = True
except ImportError:
    PYTORCH_AVAILABLE = False

# --- Общий класс приложения ---
class DrawingApp:
    def __init__(self, predictor, alphabet, title):
        self.predictor = predictor
        self.alphabet = alphabet
        self.img_size = 32
        self.canvas_size = 400
        self.brush_size = 18
        
        self.root = tk.Tk()
        self.root.title(title)
        self.root.configure(bg='#f0f0f0')
        
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0)

        self.canvas = tk.Canvas(main_frame, width=self.canvas_size, height=self.canvas_size, 
                                bg='white', cursor='pencil', highlightthickness=2)
        self.canvas.grid(row=0, column=0, rowspan=10, padx=10)
        
        results_frame = ttk.LabelFrame(main_frame, text=" Вероятности ", padding="10")
        results_frame.grid(row=0, column=1, rowspan=10, sticky='ns', padx=10)
        
        self.prob_labels = {}
        self.prob_bars = {}
        
        for i, char in enumerate(self.alphabet):
            lbl = ttk.Label(results_frame, text=f"'{char}':", font=('Courier', 12))
            lbl.grid(row=i, column=0, sticky='e', pady=2)
            bar = ttk.Progressbar(results_frame, orient='horizontal', length=150, mode='determinate')
            bar.grid(row=i, column=1, padx=5)
            percent_lbl = ttk.Label(results_frame, text="0.0%", width=8)
            percent_lbl.grid(row=i, column=2, sticky='w')
            self.prob_labels[char] = percent_lbl
            self.prob_bars[char] = bar

        controls = ttk.Frame(main_frame)
        controls.grid(row=11, column=0, columnspan=2, pady=10, sticky='ew')
        ttk.Button(controls, text='Очистить', command=self.clear_canvas).pack(side='left', padx=5)
        
        self.preview_canvas = tk.Canvas(controls, width=64, height=64, bg='black')
        self.preview_canvas.pack(side='right', padx=20)
        ttk.Label(controls, text="Вход ИИ:").pack(side='right')

        self.image = Image.new('L', (self.canvas_size, self.canvas_size), 255)
        self.draw = ImageDraw.Draw(self.image)
        
        self.canvas.bind('<B1-Motion>', self.paint)
        self.canvas.bind('<ButtonRelease-1>', lambda e: self.process_and_predict())
        self.last_x, self.last_y = None, None

    def paint(self, event):
        if self.last_x and self.last_y:
            self.canvas.create_line(self.last_x, self.last_y, event.x, event.y, 
                                   width=self.brush_size, fill='black', capstyle=tk.ROUND, smooth=tk.TRUE)
            self.draw.line([self.last_x, self.last_y, event.x, event.y], 
                           fill=0, width=self.brush_size)
        self.last_x, self.last_y = event.x, event.y

    def clear_canvas(self):
        self.canvas.delete("all")
        self.image = Image.new('L', (self.canvas_size, self.canvas_size), 255)
        self.draw = ImageDraw.Draw(self.image)
        self.last_x, self.last_y = None, None
        for char in self.alphabet:
            self.prob_labels[char].config(text="0.0%", foreground='black')
            self.prob_bars[char]['value'] = 0

    def process_and_predict(self):
        self.last_x, self.last_y = None, None
        inv_img = ImageOps.invert(self.image)
        bbox = inv_img.getbbox()
        if not bbox: return
        letter = inv_img.crop(bbox)
        w, h = letter.size
        max_dim = max(w, h)
        pad = int(max_dim * 0.3)
        container = Image.new('L', (max_dim + pad, max_dim + pad), 0)
        container.paste(letter, ((max_dim + pad - w)//2, (max_dim + pad - h)//2))
        letter_resized = container.resize((self.img_size, self.img_size), Image.Resampling.LANCZOS)
        img_array = np.array(letter_resized).astype(np.float32) / 255.0
        img_array = (img_array > 0.35).astype(np.float32)
        prev_show = letter_resized.resize((64, 64), Image.Resampling.NEAREST)
        self.ph_preview = ImageTk.PhotoImage(prev_show)
        self.preview_canvas.create_image(0, 0, anchor='nw', image=self.ph_preview)
        results = self.predictor.predict_all(img_array)
        top_char = results[0][0]
        for char, prob in results:
            percent = prob * 100
            self.prob_labels[char].config(text=f"{percent:>5.1f}%")
            self.prob_bars[char]['value'] = percent
            if char == top_char and prob > 0.1:
                self.prob_labels[char].config(foreground='green', font=('Courier', 12, 'bold'))
            else:
                self.prob_labels[char].config(foreground='black', font=('Courier', 12))

    def run(self):
        self.root.mainloop()

# ================= МЕНЮ ВЫБОРА МОДЕЛИ =================
import os

def test_model(model_type):
    if model_type == 1:
        if not os.path.exists('model_my.pkl'):
            print("Ошибка: файл model_my.pkl не найден! Сначала обучите и сохраните мою модель.")
            return
        print("=== ЗАГРУЗКА МОЕЙ МОДЕЛИ ===")
        with open('model_my.pkl', 'rb') as f:
            data = pickle.load(f)
        predictor = MyNetworkPredictor(data['weights'], data['biases'], data['sizes'], ALPHABET)
        title = "МОЯ МОДЕЛЬ (tanh, alpha=3, T0=0)"
    elif model_type == 2:
        if not os.path.exists('model_keras.h5'):
            print("Ошибка: файл model_keras.h5 не найден! Сначала обучите и сохраните Keras модель.")
            return
        print("=== ЗАГРУЗКА KERAS МОДЕЛИ ===")
        model = keras.models.load_model('model_keras.h5')
        predictor = KerasPredictor(model, ALPHABET)
        title = "KERAS МОДЕЛЬ (tanh)"
    elif model_type == 3:
        if not PYTORCH_AVAILABLE:
            print("Ошибка: PyTorch не установлен!")
            return
        if not os.path.exists('model_pytorch.pth'):
            print("Ошибка: файл model_pytorch.pth не найден! Сначала обучите и сохраните PyTorch модель.")
            return
        print("=== ЗАГРУЗКА PYTORCH МОДЕЛИ ===")
        pt_model = PyTorchNet(input_size, num_classes).to(device)
        pt_model.load_state_dict(torch.load('model_pytorch.pth', weights_only=True))
        predictor = PyTorchPredictor(pt_model, ALPHABET, device)
        title = "PYTORCH МОДЕЛЬ (tanh)"
    else:
        print("Неверный выбор!")
        return
    
    app = DrawingApp(predictor, ALPHABET, title)
    app.run()

# Запуск меню
print("\n" + "="*50)
print("ВЫБЕРИТЕ МОДЕЛЬ ДЛЯ ТЕСТИРОВАНИЯ:")
if os.path.exists('model_my.pkl'):
    print("  1 - Моя модель (tanh, alpha=3, T0=0)")
if os.path.exists('model_keras.h5'):
    print("  2 - Keras модель (tanh)")
if PYTORCH_AVAILABLE and os.path.exists('model_pytorch.pth'):
    print("  3 - PyTorch модель (tanh)")
print("="*50)

# Определяем доступные опции
available_choices = []
if os.path.exists('model_my.pkl'):
    available_choices.append('1')
if os.path.exists('model_keras.h5'):
    available_choices.append('2')
if PYTORCH_AVAILABLE and os.path.exists('model_pytorch.pth'):
    available_choices.append('3')

if not available_choices:
    print("Нет доступных моделей! Сначала обучите хотя бы одну модель.")
else:
    choice = input(f"Введите номер модели ({'/'.join(available_choices)}): ").strip()
    if choice in available_choices:
        test_model(int(choice))
    else:
        print("Неверный выбор!")



ВЫБЕРИТЕ МОДЕЛЬ ДЛЯ ТЕСТИРОВАНИЯ:
  3 - PyTorch модель (tanh)
=== ЗАГРУЗКА PYTORCH МОДЕЛИ ===
